# 1. Environment & Client Setup

## Imports

In [247]:
import sys
import os
from pathlib import Path

# Add project root to Python path so we can import config.py
project_root = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
sys.path.append(str(project_root))

import json
import boto3
from dotenv import load_dotenv
from qdrant_client import QdrantClient

import config

## Load Environment & Initialize Clients

In [248]:
# Load environment variables from .env file
load_dotenv(override=True)

# Initialize Qdrant Client
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"), 
    timeout=60,
)

# Initialize Bedrock Client
session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION", "us-east-1")
)

bedrock_client = session.client("bedrock-runtime")

## Connectivity Check

In [249]:
print("COLLECTION_NAME :", config.COLLECTION_NAME)
print("BEDROCK_MODEL_ID:", config.BEDROCK_MODEL_ID)
print("EMBEDDING_DIMENSION:", config.EMBEDDING_DIMENSION)
print("\nConfiguration and clients loaded successfully!")

COLLECTION_NAME : nifty50_financials
BEDROCK_MODEL_ID: amazon.titan-embed-text-v2:0
EMBEDDING_DIMENSION: 1024

Configuration and clients loaded successfully!


# 2. Context Retrieval & Formatting

## Embedding Function

In [250]:
import json
from botocore.exceptions import ClientError

def get_bedrock_embedding(text, client, config):
    """
    Returns embedding vector for a single text using Titan V2.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text must be a non-empty string")
    
    request_body = {
        "inputText": text,
        "dimensions": config.EMBEDDING_DIMENSION,
        "normalize": config.NORMALIZE_VECTORS
    }
    
    try:
        response = client.invoke_model(
            modelId=config.BEDROCK_MODEL_ID,
            contentType="application/json",
            accept="application/json",
            body=json.dumps(request_body)
        )
    except ClientError as e:
        raise RuntimeError(f"Error invoking Bedrock model") from e

    try:
        response_body = json.loads(response.get('body').read())
        embedding = response_body.get('embedding')
        
        if embedding is None:
            raise ValueError("No embedding returned from Bedrock")
            
        return embedding
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Error decoding JSON response") from e

## Retrieve Chunks Function

In [251]:
from qdrant_client.http.models import Filter, FieldCondition, MatchValue
from typing import Optional, Dict, Any, List
from qdrant_client.http.models import ScoredPoint

def retrieve_chunks(query:str, bedrock_client, qdrant_client, config: Any, filters:Optional[Dict[str, Any]] = None, limit=3)-> List[ScoredPoint]:
    """
    Retrieves relevant chunks from Qdrant.
    
    Parameters:
        query (str): The search query
        filters (dict, optional): Dictionary of field-value pairs to filter on.
                                  Example: {"company_name": "HDFC BANK LIMITED", "fiscal_year": "FY2026"}
        limit (int): Number of results to return
    """
    # Generate embedding for the query
    query_vector = get_bedrock_embedding(
        text=query,
        client=bedrock_client,
        config=config
    )

    conditions = []
    
    # Add dynamic filters if provided
    if filters:
        for field, value in filters.items():

            # Check the type first
            if not isinstance(value, (str, int, float, bool)):
                raise TypeError(f"Filter value for '{field}' must be primitive, got {type(value)}")

            conditions.append(
                FieldCondition(
                    key=field,
                    match=MatchValue(value=value)
                )
            )
    
    # Perform search
    try:
        results = qdrant_client.query_points(
            collection_name=config.COLLECTION_NAME,
            query=query_vector,
            query_filter=Filter(must=conditions) if filters else None,
            limit=limit,
            with_payload=True
        )
    except Exception as e:
        raise RuntimeError(f"Error querying Qdrant") from e

    return results.points

##  Format Context Function

In [252]:
def format_context(chunks):
    """
    Converts list of ScoredPoint objects into a formatted context string
    suitable for passing to an LLM.
    """

    if not chunks: 
        return "No context found"

    context_parts = []
    
    for point in chunks:
        chunk_id = point.payload.get('id', 'unknown')
        text_content = point.payload.get('text_content', '').strip()
        if not text_content:
            print(f"WARNING: Empty text for chunk {chunk_id}")
        context_parts.append(f"Source [{chunk_id}]: {text_content}")
    
    return "\n\n".join(context_parts)

# Verification

In [253]:
def display_results(results, title="Results"):
    if not results:
        print(" No results found!")
    else:
        print(f"\n=== {title} ===")
        print(f"Total results returned: {len(results)}\n")

        for i, point in enumerate(results, start=1):
            print(f"--- Result {i} ---")
            print(f"Score          : {point.score:.4f}")
            print(f"ID             : {point.id}")
            print(f"Company Name   : {point.payload.get('company_name', 'N/A')}")
            print(f"Fiscal Year    : {point.payload.get('fiscal_year', 'N/A')}")
            print(f"Quarter        : {point.payload.get('quarter', 'N/A')}")
            print(f"Text Preview   : {point.payload.get('text_content', '')[:150]}...")
            print("-" * 80)

    return

In [254]:
# Example 1: No filter (search across all companies)
results = retrieve_chunks(query="What are the total assets?", bedrock_client=bedrock_client, 
                qdrant_client=qdrant_client, config=config, limit=5)

formatted_context = format_context(results)
print("=== Formatted Context for LLM ===\n")
print(formatted_context)

display_results(results, title="Search without filters")
assert len(results) > 0, "Expected at least one result"
if results[0].score < 0.5:
    print(f" Warning: Low similarity score {results[0].score:.4f}")

=== Formatted Context for LLM ===

Source [500325_RELIANCE_1425445_25042025095716_WEB_parsed.json_chunk_bs_current_assets_current]: RELIANCE INDUSTRIES LIMITED - Current Assets (Current Year End):
- Inventories: 146,062.00
- CurrentInvestments: 118,709.00
- TradeReceivablesCurrent: 42,121.00
- CashAndCashEquivalents: 106,502.00
- BankBalanceOtherThanCashAndCashEquivalents: 0.00
- LoansCurrent: 5,182.00
- OtherCurrentFinancialAssets: 23,546.00
- CurrentFinancialAssets: 296,060.00
- CurrentTaxAssets: 0.00
- OtherCurrentAssets: 57,148.00
- CurrentAssets: 499,270.00

Source [500325_RELIANCE_1425445_25042025095716_WEB_parsed.json_chunk_bs_noncurrent_assets_current]: RELIANCE INDUSTRIES LIMITED - Non-Current Assets (Current Year End):
- PropertyPlantAndEquipment: 683,102.00
- CapitalWorkInProgress: 169,710.00
- Goodwill: 24,530.00
- OtherIntangibleAssets: 291,761.00
- IntangibleAssetsUnderDevelopment: 92,648.00
- InvestmentProperty: 0.00
- BiologicalAssetsOtherThanBearerPlants: 0.00
- Noncur

In [255]:
# Example 2: Multiple filters (Recommended for precision)

TEST_COMPANY = "HDFC BANK LIMITED"
TEST_YEAR = "FY2026"
QUARTER = "Q3"

results = retrieve_chunks(
    query="What are the total assets?", 
    bedrock_client=bedrock_client, 
    qdrant_client=qdrant_client, 
    config=config,
    filters={
        "company_name": TEST_COMPANY,
        "fiscal_year": TEST_YEAR,
        "quarter": QUARTER
    },
    limit=3
)

formatted_context = format_context(results)
print("=== Formatted Context for LLM ===\n")
print(formatted_context)

display_results(results, title="Search with multiple filters")
assert len(results) > 0, "Expected at least one result"
if results[0].score < 0.5:
    print(f" Warning: Low similarity score {results[0].score:.4f}")

=== Formatted Context for LLM ===

Source [500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_segment_assets]: HDFC BANK LIMITED - Segment Assets (Current Year End):
- SegmentAssets: 460,328,398.00
- UnAllocableAssets: 2,275,133.00
- NetSegmentAssets: 462,603,531.00

Source [500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_pnl_annual_asset_quality]: HDFC BANK LIMITED - Annual NPA and Asset Quality Metrics:
- GrossNonPerformingAssets: 0.00
- NonPerformingAssets: 0.00
- PercentageOfGrossNpa: 0.00
- PercentageOfNpa: 0.00

Source [500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_pnl_quarterly_asset_quality]: HDFC BANK LIMITED - Quarterly NPA and Asset Quality Metrics:
- GrossNonPerformingAssets: 0.00
- NonPerformingAssets: 0.00
- PercentageOfGrossNpa: 0.00
- PercentageOfNpa: 0.00

=== Search with multiple filters ===
Total results returned: 3

--- Result 1 ---
Score          : 0.2785
ID             : cb624

# 3. Financial Analyst Prompt Template

##  System Prompt Definition

In [271]:
SYSTEM_PROMPT = """You are a Senior Financial Analyst.

RULES:
1. ONLY use the provided context. NO outside knowledge.
2. REFUSAL: If the data is missing, respond ONLY with: "I cannot find this information in the provided documents."
3. PROFESSIONAL SENTENCE STRUCTURE: You must answer using full, concise sentences. Use these templates:
   - For single facts: "[Company] reported [Metric] of [Value] in [Period] [Citation]."
   - For growth/comparison: "[Company] [Metric] was [Value] in [Period A] and [Value] in [Period B], representing a growth of [Percentage] [Citations]."
4. NO PREAMBLE: Do not use phrases like "Based on...", "The document states...", or "According to the provided data...". Start the sentence with the company or the metric.
5. NO FORMATTING: No markdown tables, bullet points, or bold headers. Use plain text only.
6. NO SHOWING WORK: Do not show formulas or intermediate steps. Provide the final narrative answer only.
7. CITATIONS: Every number must be followed by its source ID in brackets.
8. TONE: Formal, objective, and professional.
"""

## Build Final Prompt Function (with Validation)

In [257]:
def build_final_prompt(query: str, context: str) -> dict:
    """
    Builds a prompt structure compatible with Bedrock Converse API.
    """
    # Input validation - Query
    if not isinstance(query, str):
        raise TypeError("Query must be a string")
    if not query or query.strip() == "":
        raise ValueError("Query cannot be empty or whitespace")
    
    # Input validation - Context
    if not isinstance(context, str):
        raise TypeError("Context must be a string")
    if not context or context.strip() == "":
        raise ValueError("Context cannot be empty. Retrieval returned no results.")
    
    # Dynamic user message construction (scalable)
    user_message = f"""Context:
{context}

Question: {query.strip()}

Please analyze the context and provide a response following the instructions above."""

    # Return structure compatible with Bedrock Converse API
    return {
        "system": [{"text": SYSTEM_PROMPT}],
        "messages": [
            {
                "role": "user",
                "content": [{"text": user_message}]
            }
        ]
    }

In [258]:
def format_chunks_to_context(chunks: list) -> str:
    """
    Converts a list of retrieved chunks into a single formatted context string.
    This is the bridge between Retrieval and Prompt Engineering.
    """
    if not isinstance(chunks, list):
        raise TypeError("chunks must be a list")
    if not chunks:
        return ""  
    
    formatted = []
    for chunk in chunks:
        if not isinstance(chunk, dict):
            continue
        chunk_id = chunk.get("chunk_id") or chunk.get("id") or "unknown"
        content = chunk.get("text") or chunk.get("content") or str(chunk)
        formatted.append(f"Source [chunk_{chunk_id}]: {content.strip()}")
    
    return "\n\n".join(formatted)

In [259]:
# Test the prompt builder
# Realistic test using actual chunk format (what your retriever would return)
test_chunks = [
    {
        "chunk_id": "789",
        "text": "HDFC Bank reported total assets of ₹1,85,432 crore as on 31st March 2024.",
        "document": "Q4_FY24_Report.pdf"
    },
    {
        "chunk_id": "790",
        "text": "This represents a 9.8% increase from the previous quarter."
    }
]

test_context = format_chunks_to_context(test_chunks)

# Now build prompt with the formatted context
prompt_payload = build_final_prompt(
    query="What were the total assets reported this quarter?",
    context=test_context
)

print("Context successfully built from chunks")
print(f"Number of chunks used: {len(test_chunks)}")
print(f"Context length: {len(test_context):,} characters")

print("Prompt successfully built for Bedrock Converse API")
print(f"System prompt length: {len(SYSTEM_PROMPT)} characters")
print(f"User message length: {len(prompt_payload['messages'][0]['content'][0]['text'])} characters")

Context successfully built from chunks
Number of chunks used: 2
Context length: 173 characters
Prompt successfully built for Bedrock Converse API
System prompt length: 833 characters
User message length: 328 characters


# 4. LLM Generation via Converse API

## Generate Answer Function

In [260]:
from botocore.exceptions import ClientError
from typing import Dict, Any

def generate_answer(
    prompt_payload: Dict[str, Any], 
    bedrock_client: Any, 
    model_id: str,
    inference_config: Dict = None
) -> str:
    """
    Generates response using Amazon Bedrock Converse API.
    """
    # Input validation
    if not isinstance(prompt_payload, dict):
        raise TypeError("prompt_payload must be a dictionary")
    if "system" not in prompt_payload or "messages" not in prompt_payload:
        raise ValueError("prompt_payload must contain 'system' and 'messages' keys")
    if not prompt_payload.get("messages"):
        raise ValueError("messages list cannot be empty")
    if bedrock_client is None:
        raise ValueError("bedrock_client cannot be None")
    if not isinstance(model_id, str) or not model_id.strip():
        raise ValueError("model_id must be a non-empty string")
    
    # Default inference config (can be overridden)
    if inference_config is None:
        inference_config = {
            "maxTokens": config.MAX_TOKENS,
            "temperature": config.TEMPERATURE
        }
    
    try:
        # Call using exact parameters that match your working CLI
        response = bedrock_client.converse(
            modelId=model_id,
            system=prompt_payload["system"],
            messages=prompt_payload["messages"],
            inferenceConfig=inference_config
        )
        
        # Clean response extraction
        try:
            text = response["output"]["message"]["content"][0]["text"]
            
            if isinstance(text, str) and text.strip():
                return text.strip()
            return "I cannot find this information in the provided documents."
            
        except (KeyError, IndexError, TypeError) as parse_err:
            raise RuntimeError(f"Bedrock response structure was unexpected or empty: {parse_err}") from parse_err
            
    except ClientError as e:
        error_code = e.response.get("Error", {}).get("Code", "Unknown")
        error_msg = e.response.get("Error", {}).get("Message", str(e))
        raise RuntimeError(f"Bedrock Converse ClientError [{error_code}]: {error_msg}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error during Bedrock Converse call: {str(e)}") from e

In [261]:
# Test the generation phase 
MODEL_ID = config.LLM_MODEL_ID

INFERENCE_CONFIG = {
    "maxTokens": config.MAX_TOKENS,
    "temperature": config.TEMPERATURE
}

try:
    ai_response = generate_answer(
        prompt_payload=prompt_payload,
        bedrock_client=bedrock_client,
        model_id=MODEL_ID,
        inference_config=INFERENCE_CONFIG
    )
    
    print("Generation completed successfully using Bedrock Converse API\n")
    print("Final AI Response:")
    print("-" * 80)
    print(ai_response)
    print("-" * 80)
    print(f"\nResponse length: {len(ai_response):,} characters")
    
except Exception as e:
    print(f"Generation failed: {str(e)}")

Generation completed successfully using Bedrock Converse API

Final AI Response:
--------------------------------------------------------------------------------
HDFC Bank reported total assets of ₹1,85,432 crore as on 31st March 2024 [chunk_789], representing a 9.8% increase from the previous quarter [chunk_790].
--------------------------------------------------------------------------------

Response length: 153 characters


# 5. End-to-End RAG Chain

# Orchestrator Function

In [262]:
def ask_financial_bot(
    question: str,
    filters: dict = None,
    bedrock_client: Any = None,
    qdrant_client: Any = None,
    config: Any = None,
    verbose: bool = False
) -> dict:
    """
    Orchestrates the full RAG pipeline.
    Uses a single 'filters' dict for easy scaling without changing function signature.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must be a non-empty string")
    if bedrock_client is None or qdrant_client is None:
        raise ValueError("Both bedrock_client and qdrant_client must be provided")
    if config is None:
        raise ValueError("config must be provided")
    
    filters = filters or {}
    if not isinstance(filters, dict):
        raise TypeError("filters must be a dictionary or None")
    
    if verbose:
        print("Step 1: Retrieving chunks...")
        print(f"   → Filters: {filters if filters else 'None'}")
    
    chunks = retrieve_chunks(
        query=question,
        bedrock_client=bedrock_client,
        qdrant_client=qdrant_client,
        config=config,
        filters=filters,
        limit=3
    )
    
    if verbose:
        print(f"   → Retrieved {len(chunks)} chunks")
    
    if not chunks:
        return {
            'answer': "I cannot find this information in the provided documents.",
            'sources': [],
            'context': ""
        }
    
    if verbose:
        print("Step 2: Formatting context...")
    
    context = format_context(chunks)

    sources = [str(point.payload.get("chunk_id") or point.payload.get("id") or 
                   point.payload.get("payload", {}).get("id", "unknown")) 
               for point in chunks]
    
    if verbose:
        print("Step 3: Building prompt and generating answer...")
    
    prompt_payload = build_final_prompt(query=question, context=context)
    answer = generate_answer(
        prompt_payload=prompt_payload,
        bedrock_client=bedrock_client,
        model_id=config.LLM_MODEL_ID
    )
    
    if verbose:
        print("✅ Orchestration completed.\n")
    
    return {
        'answer': answer,
        'sources': sources,
        'context': context
    }

In [263]:
# Test using dynamic filters dictionary (easy to extend later)
TEST_COMPANY = "HDFC BANK LIMITED"
TEST_YEAR = "FY2026"
QUARTER = "Q3"

result = ask_financial_bot(
    question="What were the total assets reported?",
    filters={
        "company_name": TEST_COMPANY,
        "fiscal_year": TEST_YEAR,
        "quarter": QUARTER
    },
    bedrock_client=bedrock_client,
    qdrant_client=qdrant_client,
    config=config,
    verbose=True
)

print("Answer:", result['answer'])
print("Sources:", result['sources'])

Step 1: Retrieving chunks...
   → Filters: {'company_name': 'HDFC BANK LIMITED', 'fiscal_year': 'FY2026', 'quarter': 'Q3'}
   → Retrieved 3 chunks
Step 2: Formatting context...
Step 3: Building prompt and generating answer...
✅ Orchestration completed.

Answer: The total net segment assets reported for HDFC Bank Limited were 462,603,531.00, comprising segment assets of 460,328,398.00 and unallocable assets of 2,275,133.00 [500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_segment_assets].
Sources: ['500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_segment_assets', '500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_pnl_annual_asset_quality', '500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_pnl_quarterly_asset_quality']


# 6. Evaluation Suite & Guardrail Verification

##  Stress Test Suite

In [ ]:
def run_stress_test_suite(ask_financial_bot, bedrock_client, qdrant_client, config):
    """
    Runs a stress test suite against the financial RAG pipeline.
    """
    test_cases = [
        {
            "name": "Direct Fact Retrieval",
            "question": "What were the Segment Assets reported by HDFC Bank in Q4 FY2026?",
            "filters": {"company_name": "HDFC BANK LIMITED", "fiscal_year": "FY2026"},
            "expected_behavior": "Should return the value 490,804,084.00 with the source ID for the Q4 segment assets chunk."
        },
        {
            "name": "Multi-Chunk Calculation",
            "question": "Compare the Net Segment Assets of HDFC Bank between Q3 FY2026 and Q4 FY2026. Calculate the percentage growth.",
            "filters": {"company_name": "HDFC BANK LIMITED", "fiscal_year": "FY2026"},
            "expected_behavior": "Should retrieve 'NetSegmentAssets' for Q3 (462,603,531.00) and Q4 (490,804,084.00), calculate the growth, and cite both chunks."
        },
        {
            "name": "The Hallucination Trap",
            "question": "What was HDFC Bank's revenue in Q1 2020?",
            "filters": {"company_name": "HDFC BANK LIMITED"},
            "expected_behavior": "Must strictly refuse with: 'I cannot find this information in the provided documents.'"
        },
        {
            "name": "No Relevant Context",
            "question": "What is the current share price of Tata Motors?",
            "filters": {"company_name": "TATA MOTORS"}, # Assuming TATA MOTORS isn't in your HDFC-only index
            "expected_behavior": "Should refuse because the company filter does not match the provided index data."
        }
    ]

    print("🚀 Starting Financial RAG Stress Test Suite\n")
    print("=" * 80)

    for i, test in enumerate(test_cases, 1):
        print(f"\nTEST {i}: {test['name']}")
        print(f"QUESTION: {test['question']}")
        print(f"EXPECTED: {test['expected_behavior']}")
        print("-" * 60)
        
        try:
            result = ask_financial_bot(
                question=test["question"],  
                bedrock_client=bedrock_client,
                qdrant_client=qdrant_client,
                config=config,
                filters=test["filters"] if "filters" in test else None,
                verbose=False
            )
            
            print("AI RESPONSE:")
            print(result['answer'])
            used_sources = [s for s in result['sources'] if s in result['answer']]
            print(f"\nSOURCES USED: {used_sources}")
            
        except Exception as e:
            print(f"ERROR: {str(e)}")
            result = {'answer': "ERROR", 'sources': []}
        
        print("=" * 80)
    
    print("\n Stress test suite completed. Review verdicts and refine grounding rules if needed.")

In [273]:
# Run the stress test (assumes clients and config are already initialized)
run_stress_test_suite(
    ask_financial_bot=ask_financial_bot,
    bedrock_client=bedrock_client,
    qdrant_client=qdrant_client,
    config=config
)

🚀 Starting Financial RAG Stress Test Suite


TEST 1: Direct Fact Retrieval
QUESTION: What were the Segment Assets reported by HDFC Bank in Q4 FY2026?
EXPECTED: Should return the value 490,804,084.00 with the source ID for the Q4 segment assets chunk.
------------------------------------------------------------
AI RESPONSE:
HDFC Bank reported Segment Assets of 490,804,084.00 in Q4 FY2026 [500180_INTEGRATED_FILING_BANKING_1654391_18042026073048_WEB_parsed.json_chunk_segment_assets].

SOURCES USED: ['500180_INTEGRATED_FILING_BANKING_1654391_18042026073048_WEB_parsed.json_chunk_segment_assets']

VERDICT: [   PASS   ] / [   FAIL   ]

TEST 2: Multi-Chunk Calculation
QUESTION: Compare the Net Segment Assets of HDFC Bank between Q3 FY2026 and Q4 FY2026. Calculate the percentage growth.
EXPECTED: Should retrieve 'NetSegmentAssets' for Q3 (462,603,531.00) and Q4 (490,804,084.00), calculate the growth, and cite both chunks.
------------------------------------------------------------
AI RESPONSE: